# Compact Attention


In [1]:
import torch
import torch.nn as nn

## MHA
Multi-head attention. The original version of transformer paper.

$$ Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d}})V $$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_head):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_head = num_head
        self.head_dim = hidden_size // num_head

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)

        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, X:torch.Tensor, mask=None):
        # input X: batch_size * seq_len * hidden_size
        batch_size, seq_len, _ = X.shape()

        # transform size into (batch_size, num_head, seq_len, head_dim)
        query = self.W_q(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2)
        key = self.W_k(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2)
        # todo: rope
        value = self.W_v(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2)

        # calculate attention score for every head
        p = torch.matmul(query, key.transpose(-2, -1))  / (self.head_dim ** 0.5)

        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))

        p = torch.softmax(p, dim=-1) # batch_size, num_head, seq_len, seq_len
        o = torch.matmul(p, value) # batch_size, num_head, seq_len, head_dim

        # concat
        o = o.transpose(1,2).reshape(batch_size, seq_len, self.hidden_size)

        return self.out_proj(o)
        

## MQA

Multi-Query attention. Several query head with only **one** KV head.

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, hidden_size, num_head):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_head = num_head
        self.head_dim = hidden_size // num_head

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, self.head_dim)
        self.W_v = nn.Linear(hidden_size, self.head_dim)

        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, X: torch.Tensor, mask=None):
        batch_size, seq_len, _ = X.shape()

        query = self.W_q(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1,2) # batch_size, num_head, seq_len, head_dim
        key = self.W_k(X).unsqueeze(1).expand(-1, self.num_head, -1, -1)
        value = self.W_v(X).unsqueeze(1).expand(-1, self.num_head, -1, -1)

        p = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))
        p = torch.softmax(p, dim=-1)
        o = torch.matmul(p, value)

        # concat
        o = o.transpose(1,2).reshape(batch_size, seq_len, self.hidden_size)
        return self.out_proj(o)

## GQA
Group Query Attetion. Group several query heads with one KV head, but can have multiple KV head in LLM.

In [ ]:
class GroupQueryAttention(nn.Module):
    def __init__(self, hidden_size, num_head, group_size):
        # group_size: the number of query head in one group(with one KV head)
        super().__init__()

        assert hidden_size % num_head == 0, "hidden_size must be dividable with num_head"
        assert num_head % group_size == 0, "num_head must be dividable with group_size"

        self.hidden_size = hidden_size
        self.num_head = num_head
        self.head_dim = hidden_size // num_head
        self.num_kv_head = num_head // group_size
        self.group_size = group_size

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, self.num_kv_head * self.head_dim)
        self.W_v = nn.Linear(hidden_size, self.num_kv_head * self.head_dim)

        self.out_proj = nn.Linear(hidden_size, hidden_size)

    def forward(self, X: torch.Tensor, mask=None):
        batch_size, seq_len, _ = X.shape()

        query = self.W_q(X).view(batch_size, seq_len, self.num_head, self.head_dim).transpose(1, 2)
        key = self.W_k(X).view(batch_size, seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)
        key = key.unsqueeze(2).expand(-1,-1, self.group_size, -1, -1).reshape(batch_size, self.num_head, seq_len, self.head_dim)
        value = self.W_v(X).view(batch_size, seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)
        value = value.unsqueeze(2).expand(-1,-1, self.group_size, -1, -1).reshape(batch_size, self.num_head, seq_len, self.head_dim)

        p = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            p = p.masked_fill(mask, self.float('-inf'))
        p = torch.softmax(p, dim=-1)
        o = torch.matmul(p, value)

        # concat
        o = o.transpose(1,2).reshape(batch_size, seq_len, self.hidden_size)
        return self.out_proj(o)


## MLA
![Compact Attentions](../figs/2.2-MLA.png)
*This picture is the comparison of different compact attention mechanisms. (From DeepSeek-v2 paper, see **Reference 2**)*

## References

1. (paper) Attention is all you need, https://arxiv.org/pdf/1706.03762
2. (paper) DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model, https://arxiv.org/pdf/2405.04434